In [ ]:
# --- Cell 1: 核心配置、授权与目录规范 ---
import os
from pathlib import Path

# 1. 身份授权 (请确保 Token 正确)
TUSHARE_TOKEN = 'your token'

# 2. 目录规范
BASE_DIR = Path(os.getcwd())
RAW_DIR = BASE_DIR / 'data_raw'      
CACHE_DIR = BASE_DIR / 'data_cache'  
OUT_DIR = BASE_DIR / 'data_output'   
LOG_DIR = BASE_DIR / 'log'           

# 自动创建目录
for d in [RAW_DIR, CACHE_DIR, OUT_DIR, LOG_DIR]:
    d.mkdir(parents=True, exist_ok=True)


In [ ]:
# --- Cell 2: 接口配置字典 ---
# split_mode 说明:
# - None: 全量下载 (如股票列表)
# - 'quarter': 按季度拆分 (适合交易日频率的大表)
# - 'single_file': 所有日期存入一个文件 (适合数据量中的表)
# - 'by_stock': 按股票代码循环 (适合财务报表 07, 08, 09, 10 等强制要求 ts_code 的接口)
API_CONFIG = {
    # === P0 基础表 ===
    '01_stock_basic': {'api': 'stock_basic', 'date_param': None, 'split_mode': None, 'fields': 'ts_code,symbol,name,area,industry,market,exchange,list_status,list_date,delist_date,is_hs'},
    '02_trade_cal': {'api': 'trade_cal', 'date_param': None, 'split_mode': None, 'fields': 'exchange,cal_date,is_open,pretrade_date'},
    
    # === P1 交易/行情类 (按日期增量) ===
    '03_daily': {'api': 'daily', 'date_param': 'trade_date', 'split_mode': 'quarter', 'fields': 'ts_code,trade_date,open,high,low,close,pre_close,change,pct_chg,vol,amount'},
    '04_adj_factor': {'api': 'adj_factor', 'date_param': 'trade_date', 'split_mode': 'quarter', 'fields': 'ts_code,trade_date,adj_factor'},
    '05_index_daily': {'api': 'index_daily', 'date_param': 'trade_date', 'split_mode': 'single_file', 'extra_params': {'ts_code': '000001.SH,399001.SZ,000300.SH,000905.SH,000852.SH'}, 'fields': 'ts_code,trade_date,open,high,low,close,pre_close,change,pct_chg,vol,amount'},
    '06_daily_basic': {'api': 'daily_basic', 'date_param': 'trade_date', 'split_mode': 'quarter', 'fields': 'ts_code,trade_date,close,turnover_rate,turnover_rate_f,volume_ratio,pe,pe_ttm,pb,ps,ps_ttm,dv_ttm,total_mv,circ_mv'},

    # === P2 财务报表类 (修复版：改用 by_stock 模式) ===
    '07_income': {'api': 'income', 'date_param': 'ann_date', 'split_mode': 'by_stock', 'fields': 'ts_code,ann_date,f_ann_date,end_date,report_type,total_revenue,revenue,n_income,n_income_attr_p,basic_eps'},
    '08_fina_indicator': {'api': 'fina_indicator', 'date_param': 'ann_date', 'split_mode': 'by_stock', 'fields': 'ts_code,ann_date,end_date,roe,roe_dt,roa,grossprofit_margin,netprofit_margin,debt_to_assets,ocf_to_or'},
    '09_forecast': {'api': 'forecast', 'date_param': 'ann_date', 'split_mode': 'by_stock', 'fields': 'ts_code,ann_date,end_date,type,p_change_min,p_change_max,net_profit_min,net_profit_max,last_parent_net,first_ann_date,summary'},
    '10_express': {'api': 'express', 'date_param': 'ann_date', 'split_mode': 'by_stock', 'fields': 'ts_code,ann_date,end_date,revenue,operate_profit,total_profit,n_income,total_assets,diluted_roe,yoy_net_profit'},

    # === P3 其他大表与局部表 ===
    '11_disclosure_date': {'api': 'disclosure_date', 'date_param': 'date', 'split_mode': 'single_file', 'fields': 'ts_code,end_date,pre_date,actual_date,modify_date'},
    '12_suspend_d': {'api': 'suspend_d', 'date_param': 'trade_date', 'split_mode': 'quarter', 'fields': 'ts_code,trade_date,suspend_timing,suspend_type'},
    '13_stk_limit': {'api': 'stk_limit', 'date_param': 'trade_date', 'split_mode': 'quarter', 'fields': 'ts_code,trade_date,up_limit,down_limit'},
    '14_sw_member': {'api': 'index_member', 'date_param': None, 'split_mode': None, 'fields': 'index_code,index_name,con_code,con_name,in_date,out_date'},
    '15_moneyflow': {'api': 'moneyflow', 'date_param': 'trade_date', 'split_mode': 'quarter', 'fields': 'ts_code,trade_date,buy_sm_amount,buy_md_amount,buy_lg_amount,buy_elg_amount,net_mf_amount'},
    '16_margin': {'api': 'margin', 'date_param': 'trade_date', 'split_mode': 'quarter', 'fields': 'trade_date,ts_code,rzye,rqye,rzmre,rzche,rqmcl'},
    '17_hk_hold': {'api': 'hk_hold', 'date_param': 'trade_date', 'split_mode': 'quarter', 'fields': 'ts_code,trade_date,vol,ratio,exchange'},
    '18_stk_holdernumber': {'api': 'stk_holdernumber', 'date_param': 'ann_date', 'split_mode': 'by_stock', 'fields': 'ts_code,ann_date,end_date,holder_num'},
    '19_top10_holders': {'api': 'top10_holders', 'date_param': 'ann_date', 'split_mode': 'by_stock', 'fields': 'ts_code,ann_date,end_date,holder_name,hold_amount,hold_ratio'},
    '20_block_trade': {'api': 'block_trade', 'date_param': 'trade_date', 'split_mode': 'quarter', 'fields': 'ts_code,trade_date,price,vol,amount,buyer,seller'},
    '21_index_weight': {'api': 'index_weight', 'date_param': 'trade_date', 'split_mode': 'single_file', 'extra_params': {'index_code': '000001.SH,399001.SZ,000300.SH,000905.SH,000852.SH'}, 'fields': 'index_code,con_code,trade_date,weight'},
    '22_macro': {'api': 'shibor', 'date_param': 'date', 'split_mode': 'single_file', 'fields': 'date,shibor,shibor_quote,shibor_ma'}
}


In [ ]:
# --- Cell 3: 运行参数 ---
START_DATE = '20240101' # 训练起步日期可修改
SLEEP_TIME = 0.20      # 5000 积分大佬专属加速频率